- ### Template

1. [Function template](#function-template)

2. [Class template](#class-template)

3. [Multi-type template](#multi-type-template)

---

- ### Function template

In [1]:
# Setup for oneline command %%cpp
import os, tempfile, subprocess
from IPython.core.magic import register_cell_magic
import shlex

@register_cell_magic
def cpp(line, cell):
    """
    Usage:
    %%cpp -i "input for cin" -- arg1 arg2 ...
    """
    tokens = shlex.split(line)
    input_data = None
    run_args = []

    # Parse stdin input
    if "-i" in tokens:
        idx = tokens.index("-i")
        if idx + 1 < len(tokens):
            input_data = tokens[idx + 1]

    # Parse program arguments after --
    if "--" in tokens:
        idx = tokens.index("--")
        run_args = tokens[idx + 1:]

    # Write temp C++ file
    with tempfile.NamedTemporaryFile(suffix=".cpp", delete=False, mode="w") as tmp_cpp:
        tmp_cpp.write(cell)
        cpp_path = tmp_cpp.name
    exe_path = cpp_path[:-4] + ".exe"

    try:
        # Compile
        compile_proc = subprocess.run(
            ["g++", "-std=c++23", "-O2", "-Wall", cpp_path, "-o", exe_path],
            capture_output=True,
            text=True
        )
        if compile_proc.returncode != 0:
            print("❌ Compilation failed:\n", compile_proc.stderr)
            return

        # Run program
        run_proc = subprocess.run(
            [exe_path] + run_args,
            input=input_data,      # feed stdin here
            capture_output=True,
            text=True
        )
        if run_proc.stdout:
            print(run_proc.stdout, end="")
        if run_proc.stderr:
            print("⚠️ Runtime error:\n", run_proc.stderr)

    finally:
        for f in (cpp_path, exe_path):
            try: os.remove(f)
            except: pass

In [5]:
%%cpp
#include <iostream>
using namespace std;

template<typename T> // works for any type T that supports +
T add(T a, T b) {
    return a + b;
}

int main() {
    int x = 5, y = 10;
    double a = 5.5, b = 10.5;
    cout << "Sum: " << add(x, y) << endl; // works with int
    cout << "Sum: " << add(a, b) << endl; // works with double
}

Sum: 15
Sum: 16


---

- ### Class template

In [14]:
%%cpp
#include <iostream>
#include <string>
using namespace std;

template<class a> // class can be used instead of typename
class Box {
    a value;
public:
    Box(a v) : value(v) {}
    a getValue() { return value; }
};

int main() {
    Box<int> intBox(123);
    Box<string> strBox("Hello, Templates!");
    cout << "Integer Box: " << intBox.getValue() << endl;
    cout << "String Box: " << strBox.getValue() << endl;
}

Integer Box: 123
String Box: Hello, Templates!


---

- #### Multi-type template

In [15]:
%%cpp
#include <iostream>
using namespace std;

template<class S, class T>
class Pair {
    S first;
    T second;
    public:
    Pair(S f, T s) : first(f), second(s) {}
    void display() {
        cout << "First: " << first << ", Second: " << second << endl;
    }
};

int main() {
    Pair<int, double> p1(42, 3.14);
    Pair<string, int> p2("Age", 30);
    p1.display();
    p2.display();
}

First: 42, Second: 3.14
First: Age, Second: 30
